# 18 · Final test v4 (run6 on Juice Shop + DVWA, real tools, a real public log)

This is the locked final test of the whole system, frozen at tag `v4.0-final-test`. It runs everything the backend
runs on data none of the code was built or tuned on:

- **run6** (seed 6, both sites): made with generator v4, which adds 7 newer attack families (SSRF, LDAP, NoSQL,
  SSTI, XXE, Log4Shell, XPath) to the older ones, plus 60 campaign attackers. The files were hashed right after they
  were made and sealed without being opened. The v4 payloads were never run through the detector before this run.
- **tools3**: real tools against our own lab: ffuf (content discovery) and XSStrike (XSS) on both sites, sealed unopened.
  In a tool log every request from the tools container is an attack and every other IP is a normal visitor.
- **zanbil**: a real, public access log of a live online shop (Kaggle, eliasdabbas/web-server-access-logs). It has no
  labels, so it measures false alarms only: what the system flags on ordinary real traffic, and why.

Tables: detection per detector (with the **rules-only** baseline), recall per attack type, sessions, attack type,
stage, next step, defence, the unusual mark, then tools, the public log and the same logs without bodies.

**Rules for this notebook:**
- The code is frozen first (tag `v4.0-final-test`) and the commit is printed below; it must say `(clean)`.
- With `RUN_FINAL = False` it runs on development data (v4dev: a small v4 run on both sites, 10 campaigns), only to test
  the notebook. The next-step model and the block rules were trained on campaign data, so dev numbers mean little.
- The SHA-256 of every file read is checked first against the hash lists written when the files were made
  (`run6_hashes.txt`, `tools3_hashes.txt`, and the zanbil hash below). If any file changed, the notebook stops before
  any result is shown.
- `RUN_FINAL = True` is run **once**. Whatever comes out goes in the report. Nothing is changed after seeing it.

In [ ]:
RUN_FINAL = False   # set to True ONCE, after the code is frozen

# Filled in before the final run, never after:
TOOLS3_IP = '172.19.0.7'  # tools container IP in every earlier tool run; container deleted, checked below
ZANBIL_SHA256 = '552FEDF1865952F600D72675D88000C63162D63F080E52F667282BB689D75112'  # sent by Safi right after download, 2026-11-01
ZANBIL_LINES = 200_000  # the first N lines of the zanbil log are used (the whole file is ~10M lines)

In [ ]:
import hashlib, re, subprocess, sys, tempfile
from itertools import islice
from pathlib import Path, PureWindowsPath
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from backend.app.log_parser import decode, parse_log
from eval.final_test import (defence_summary, investigate, next_step_summary, recall_by_type, run, run_tool,
                             stage_summary, summary, type_accuracy, unusual_summary)

commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
tag = subprocess.run(['git', 'describe', '--tags', '--exact-match'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain', '--', 'ml', 'eval', 'backend'], cwd=ROOT,
                       capture_output=True, text=True).stdout.strip()
print('code commit:', commit, tag or '(no tag)', '(uncommitted changes in ml/eval/backend!)' if dirty else '(clean)')

D = ROOT / 'data'
# name: (log, request labels, session labels)
if RUN_FINAL:
    if TOOLS3_IP is None or ZANBIL_SHA256 is None:
        raise RuntimeError('Set TOOLS3_IP and ZANBIL_SHA256 first.')
    RUNS = {'Juice Shop (run6)': (D / 'raw' / 'juiceshop_generated_run6.log', D / 'labels' / 'requests_run6.csv',
                                  D / 'labels' / 'sessions_run6.csv'),
            'DVWA (run6)': (D / 'raw' / 'dvwa_generated_run6.log', D / 'labels' / 'requests_dvwa_run6.csv',
                            D / 'labels' / 'sessions_dvwa_run6.csv')}
    TOOLS = {f'{tool} on {site}': D / 'raw' / f'tool_{tool}_{site}.log'
             for tool in ['ffuf', 'xsstrike'] for site in ['juiceshop', 'dvwa']}
    ZANBIL = D / 'raw' / 'zanbil_access.log'
else:
    RUNS = {'Juice Shop (v4dev)': (D / 'raw' / 'juiceshop_generated_v4dev.log', D / 'labels' / 'requests_v4dev.csv',
                                   D / 'labels' / 'sessions_v4dev.csv'),
            'DVWA (v4dev)': (D / 'raw' / 'dvwa_generated_v4dev.log', D / 'labels' / 'requests_dvwa_v4dev.csv',
                             D / 'labels' / 'sessions_dvwa_v4dev.csv')}
    TOOLS = {}   # filled in section 7 from a dev log
    ZANBIL = D / 'raw' / 'public_apache_logs.log'   # stand-in: another public log, only to test the code

# Fingerprint of every file read. In the final run it must match the hash lists written when the files were made.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest().upper()   # upper: same as Get-FileHash

def read_hashes(path):
    raw = Path(path).read_bytes()   # missing: ask for it, never skip the check
    text = raw.decode('utf-16') if raw[:2] in (b'\xff\xfe', b'\xfe\xff') else raw.decode('utf-8-sig')
    return {PureWindowsPath(p.strip()).name: h.upper() for h, p in re.findall(r'\b([0-9A-Fa-f]{64})\s+(.+)', text)}

files = [f for fs in RUNS.values() for f in fs] + list(TOOLS.values()) + ([ZANBIL] if ZANBIL.exists() else [])
hashes = {f.name: sha256(f) for f in files}
for file_name, h in hashes.items():
    print(h, file_name)
if RUN_FINAL:
    expected = {ZANBIL.name: ZANBIL_SHA256.upper()}
    for hash_list in ['run6_hashes.txt', 'tools3_hashes.txt']:
        expected.update(read_hashes(D / 'labels' / hash_list))
    changed = [f for f, h in hashes.items() if expected.get(f) != h]
    if changed:
        raise RuntimeError(f'Not as in the hash lists (changed, or missing from the lists): {changed}. '
                           'Stop here, do not look at any result.')
    print('all files match the hash lists')

results = {}
for name, (log, labels, _) in RUNS.items():
    req, sessions, report = run(log, labels)
    req, sessions = investigate(req, sessions)
    results[name] = (req, sessions)
    print(name, report)

## 1. Requests: how many attacks are caught, and how many false alarms

`rules` is the baseline: fixed attack patterns, no machine learning. `+ sessions` adds the session layer
(brute force, enumeration, scanning). A false alarm "on normal users" is a normal request in a normal user's session;
the others are ordinary pages an attacker loads between two attacks (inside a session that really is an attacker's).

In [ ]:
pd.concat({name: summary(req) for name, (req, _) in results.items()})

## 2. Recall by attack type

The newer families (ssrf, xxe, ssti, nosql, log4shell, ldap, xpath) are the ones the v4 generator added.

In [ ]:
pd.concat({name: recall_by_type(req) for name, (req, _) in results.items()})

## 3. Sessions: is every attacker found, and no normal user flagged?

`critical` = high risk and the attacker reached exploitation or data theft.

In [ ]:
pd.concat({name: pd.crosstab(s['truth'], s['risk_level']) for name, (_, s) in results.items()})

## 4. Attack type of the caught payload attacks

In [ ]:
for name, (req, _) in results.items():
    table, acc = type_accuracy(req)
    print(f'{name}: {acc:.1%} correct')
    display(table)

## 5. Stage, next step and defence

**Stage**: the furthest stage each session reached. Normal users should have none.

**Next step**: only the campaign sessions (attackers that do several kinds of attack one after the other). *Order read exactly* = the steps read from the log are the steps the generator ran. Top-1 / top-3 = how often the real next step is the model's first guess / among its first three, on the true order and on the order read from the log.

**Defence**: the right action comes from the labels: any payload attack -> block, only brute force, enumeration or scanning -> rate limit, normal -> allow. The two mistakes that matter are an attacker allowed and a normal user blocked.

In [ ]:
pd.concat({name: stage_summary(s) for name, (_, s) in results.items()})

In [ ]:
pd.DataFrame({name: next_step_summary(req, RUNS[name][2]) for name, (req, _) in results.items()})

In [ ]:
pd.DataFrame({name: defence_summary(req, s) for name, (req, s) in results.items()})

## 6. The unusual-request mark

Review only: it never changes a label or a risk. It marks the most unusual requests of each log (Isolation Forest fitted on that log). The rows that matter: how many normal requests it marks, and whether it marks any attack the detector and the session layer missed.

In [ ]:
pd.DataFrame({name: unusual_summary(req) for name, (req, _) in results.items()})

## 7. Real tools (ffuf and XSStrike)

Each tool log is one run of one tool against one site. Requests from the tools container are attacks; any other IP
in the file (for example the browser on the host) is a normal visitor. If the tools container IP does not appear in a
file, the notebook stops and prints only the IPs and their request counts.

In [ ]:
if not RUN_FINAL:
    # Code check only: a dev log cut down to the normal users plus one attacker IP, in the shape of a tool log.
    dev_log, dev_labels, _ = RUNS['DVWA (v4dev)']
    lab = pd.read_csv(dev_labels, keep_default_na=False)
    normal_ips = set(lab['ip']) - set(lab.loc[lab['label'] != 'normal', 'ip'])
    attacker = lab.loc[lab['attack_type'] == 'xss', 'ip'].value_counts().index[0]
    lines = [l for l in open(dev_log, encoding='utf-8') if l.split(' ', 1)[0] in normal_ips | {attacker}]
    tmp = Path(tempfile.mkdtemp()) / 'tool_dev_check.log'
    tmp.write_text(''.join(lines), encoding='utf-8')
    TOOLS = {'dev check (one xss attacker on DVWA v4dev)': tmp}
    TOOL_ATTACKER = attacker
else:
    TOOL_ATTACKER = TOOLS3_IP

for name, path in TOOLS.items():
    ips = pd.Series([r['ip'] for r in parse_log(decode(Path(path).read_bytes()))[0]]).value_counts()
    if TOOL_ATTACKER not in ips.index:
        print(name, ips.to_dict())
        raise RuntimeError(f'{TOOL_ATTACKER} is not in {name}. Stop here; find the right tools container IP.')

rows, tool_results = {}, {}
for name, path in TOOLS.items():
    req, sessions, report = run_tool(path, attacker_ip=TOOL_ATTACKER)
    req, sessions = investigate(req, sessions)
    tool_results[name] = (req, sessions)
    tool = req['ip'] == TOOL_ATTACKER
    tool_sessions = sessions[sessions['session'].isin(req.loc[tool, 'session'])]
    caught = req.loc[tool & (req['model + sessions'] == 1)]
    rows[name] = {**report,
                  'caught (model + sessions)': round(float(req.loc[tool, 'model + sessions'].mean()), 3),
                  'caught (model)': round(float(req.loc[tool, 'model'].mean()), 3),
                  'caught (rules + sessions)': round(float(req.loc[tool, 'rules + sessions'].mean()), 3),
                  'caught (rules)': round(float(req.loc[tool, 'rules'].mean()), 3),
                  'false alarms on other visitors': int(req.loc[~tool, 'model + sessions'].sum()),
                  'tool sessions': len(tool_sessions),
                  'tool risk levels': ', '.join(sorted(set(tool_sessions['risk_level']))),
                  'tool actions': ', '.join(sorted(set(tool_sessions['action']))),
                  'tool stages': ', '.join(sorted(set(tool_sessions['stage'].replace('', 'none')))),
                  'types given': ', '.join(f'{t} {n}' for t, n in caught['type_pred'].replace('', 'session only')
                                           .value_counts().items()),
                  'tool requests marked unusual': int(req.loc[tool, 'unusual'].sum())}
pd.DataFrame(rows).T

## 8. A real public log (zanbil): false alarms on real traffic

The log has no labels, so every request is treated as normal and every flag is listed for review. Only the first
`ZANBIL_LINES` lines are used, so the run fits on a laptop. A flag is not automatically wrong: a real shop log also
holds real scanners and bots. Section 9 writes down, for each kind of flag, whether it is a real probe or a mistake.

In [ ]:
if ZANBIL.exists():
    sample = Path(tempfile.mkdtemp()) / 'public_sample.log'
    with open(ZANBIL, 'rb') as src, open(sample, 'wb') as dst:
        dst.writelines(islice(src, ZANBIL_LINES))
    zreq, zses, zrep = run_tool(sample, attacker_ip='none')   # no labels: everything counted normal
    zreq, zses = investigate(zreq, zses)
    print(zrep)
    print('requests:', len(zreq), '| IPs:', zreq['ip'].nunique(), '| sessions:', len(zses))
    flags = pd.Series({'flagged by model': int(zreq['pred'].sum()),
                       'flagged by sessions only': int(((zreq['pred'] == 0) & (zreq['session_attack'] == 1)).sum()),
                       'flagged (model + sessions)': int(zreq['model + sessions'].sum()),
                       'flagged by rules': int(zreq['rules'].sum()),
                       'share of requests flagged (model + sessions)': round(float(zreq['model + sessions'].mean()), 4),
                       'marked unusual': int(zreq['unusual'].sum())})
    display(flags)
    display(pd.crosstab(zses['risk_level'], zses['action']))
else:
    print('public log not on this machine: section skipped')

In [ ]:
if ZANBIL.exists():
    pd.set_option('display.max_colwidth', 120)
    print('flagged sessions (medium and above):')
    display(zses[zses['risk_level'].isin(['medium', 'high', 'critical'])]
            [['ip', 'requests', 'attack_requests', 'risk_level', 'action', 'stage', 'reasons']].head(50))
    print('requests flagged by the model:')
    display(zreq.loc[zreq['pred'] == 1, ['ip', 'method', 'path', 'status', 'type_pred']].head(100))
    print('session-only flags, by behaviour:')
    display(zreq.loc[(zreq['pred'] == 0) & (zreq['session_attack'] == 1), 'behaviour'].value_counts())

## 9. Same logs without request bodies

Most real access logs do not store the body. This shows what is lost (attacks sent in a POST body) and checks the false alarms.

In [ ]:
pd.concat({name: summary(run(log, labels, drop_body=True)[0]) for name, (log, labels, _) in RUNS.items()}
          | {name: summary(run_tool(path, attacker_ip=TOOL_ATTACKER, drop_body=True)[0]) for name, path in TOOLS.items()})

## 10. What this shows

*(Write this after the one final run, from the numbers above. Do not change the code or rerun.)*